In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold
from lightgbm import LGBMClassifier
import optuna
import warnings
warnings.filterwarnings('ignore')


SEED = 42

In [ ]:
ROOT = Path.cwd().parent.resolve()

TRAIN = ROOT / "data/train.csv"
TEST = ROOT / "data/test.csv"
OUT = ROOT / "outputs"
Path.mkdir(OUT , exist_ok=True)
OUT = OUT / "lightgbm"
Path.mkdir(OUT , exist_ok=True)

train_df = pd.read_csv(str(TRAIN), sep=',')
test_df = pd.read_csv(str(TEST), sep=',')

X = train_df.drop(columns=["id", "Heart Disease"])
le = LabelEncoder()
y = le.fit_transform(train_df["Heart Disease"])
X_test = test_df.drop(columns="id")

cat_features = X.select_dtypes(include=["object", "category"]).columns.tolist()

In [ ]:
def objective(trial):
    params = {
        #--- Search Space ---#
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.8, log=True),
        "num_leaves": trial.suggest_int("num_leaves", 128, 256),
        "max_depth": trial.suggest_int("max_depth", 1, 5),
        "min_data_in_leaf": trial.suggest_int("min_data_in_leaf", 20, 200),
        "feature_fraction": trial.suggest_float("feature_fraction", 0.4, 1.0),
        "bagging_fraction": trial.suggest_float("bagging_fraction", 0.5, 1.0),
        "bagging_freq": trial.suggest_int("bagging_freq", 1, 10),
        "lambda_l1": trial.suggest_float("lambda_l1", 1e-8, 10.0, log=True),
        "lambda_l2": trial.suggest_float("lambda_l2", 1e-8, 10.0, log=True),
    }

    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
    oof = np.zeros(len(X))

    for train_idx, val_idx in skf.split(X, y):
        X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
        y_train, y_val = y[train_idx], y[val_idx]

        model = LGBMClassifier(
            objective="binary",
            metric="auc",
            boosting_type="gbdt",
            verbosity=-1,
            random_state=SEED,
            early_stopping_rounds=100,
            **params
        )

        model.fit(
            X_train, y_train,
            eval_set=[(X_val, y_val)],
        )

        oof[val_idx] = model.predict_proba(X_val)[:, 1]

    return roc_auc_score(y, oof)

In [ ]:
study = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=SEED) 
)

study.optimize(objective, n_trials=70)

In [ ]:
best_params = study.best_params
seeds = [1, 42, 254, 2126]
tests = []
oofs = []

for seed in seeds:
    params = {**best_params, "random_state": seed}

    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=seed)

    oof_seed = np.zeros(len(X))
    test_seed = np.zeros(len(X_test))

    for train_idx, val_idx in skf.split(X, y):
        X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
        y_train, y_val = y[train_idx], y[val_idx]

        model = LGBMClassifier(
            **params,
            objective="binary",
            metric="auc",
            boosting_type="gbdt",
            verbosity=-1,
            early_stopping_rounds=100,
        )
        model.fit(
            X_train, y_train,
            eval_set=[(X_val, y_val)],
        )

        oof_seed[val_idx] += model.predict_proba(X_val)[:, 1]
        test_seed += model.predict_proba(X_test)[:, 1] / skf.n_splits

    oofs.append(oof_seed)
    tests.append(test_seed)

roc_auc_score(y, np.vstack(oofs).mean(axis=0))

In [ ]:
final_oofs = np.vstack(oofs).mean(axis=0)
final_preds = np.vstack(tests).mean(axis=0)

np.save(OUT / "light_oof.npy", final_oofs)
np.save(OUT / "light_test.npy", final_preds)

submission = pd.DataFrame({
    "id": test_df["id"],
    "Heart Disease": final_preds
})
submission.to_csv(OUT / "light_submission.csv", index=False)
submission